<a href="https://colab.research.google.com/github/JuliaMcPhillips/ds2002-fa26/blob/main/Copy_of_2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units sold: {total_units}")

Total revenue: $8,520.00
Total units sold: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).to_frame()
by_category['pct_of_total'] = (by_category['revenue'] / total_revenue * 100).round(1)
by_category

,revenue,pct_of_total
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
by_vendor = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).sort_values('mean', ascending=False)
by_vendor

,mean,count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = by_category.loc['Merch', 'pct_of_total']
print(f"Merch share of revenue: {merch_share}%")

Merch share of revenue: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(joined) == len(df), "row count changed!"
assert abs(joined['revenue'].sum() - total_revenue) < 0.01, "revenue changed!"

unmatched_ids = df.loc[~df['vendor_id'].isin(vendor_names['vendor_id']), 'vendor_id'].unique()
print("Unmatched vendor id(s):", unmatched_ids)
print("Rows with no vendor_name:", joined['vendor_name'].isna().sum())

Unmatched vendor id(s): ['V-18']
Rows with no vendor_name: 108


**The unmatched vendor, and what I did about it:** _I noticed V-18 isn't in the lookup table at all after the left join, 108 rows, came back with a null vendor_name. Rather than drop those rows, which would silently remove over a quarter of all revenue from the report, I filled the nulls with a placeholder built from the vendor_id itself. This is written generally, not as a one-off patch for V-18 so it would correctly label any vendor missing from the lookup, including if a future game had two or three unmatched vendors instead of just one._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [9]:
pivot = pd.pivot_table(
    df, index='vendor_id', columns='category', values='revenue',
    aggfunc='sum', fill_value=0, margins=True, margins_name='Total'
)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [10]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell the vendors that food is the main revenue driver since its half of all revenue (4,293 of 8,520), but RainGear is barely contributing at 10.6% (901.50). I'd tell vendors to cut RainGear inventory next game, since it's not pulling weight. I'd also flag V-01 to the vendors as a bench mark, its average order value of 22.60 is the highest of the four, about 11% above V-10's $20.31, but this comparison may not be statistically meaningful as I explain below.

b) Q3 (average order revenue by vendor) is the least trustworthy. The four averages are only about 2 dollars  apart (20.31 dollars to 22.60 dollars), and each one is based on roughly 100 orders where the price per order can swing a lot (anywhere from 4.50 dollars to 72 dollars). With that much variation in individual orders and only only about 100 orders to average over, I dont think a 2 dollar difference is enough to say V-01 is actually "better." It could easily just be luck in which orders happened to land with which vendor. I'd want to see this hold up over a few more games before trusting the ranking.